# 03 - Filtered-x Practical ANC

**Standalone demo notes**

The Filtered-x family is the practical ANC step. In software denoising notebooks, the adaptive filter output can be subtracted directly. In physical ANC, the controller drives a loudspeaker, and the anti-noise reaches the error microphone only after passing through the secondary path $S(z)$.

Primary disturbance:

$$
d[n] = p[n] * x[n]
$$

Secondary-path output and residual:

$$
y_s[n] = s[n] * y[n], \qquad e[n] = d[n] + y_s[n]
$$

Filtered reference:

$$
x'[n] = \hat{s}[n] * x[n]
$$

FxLMS update:

$$
\mathbf{w}_{n+1} = \mathbf{w}_n - \mu e[n]\mathbf{x}'[n]
$$

FxNLMS update:

$$
\mathbf{w}_{n+1} =
\mathbf{w}_n -
\frac{\tilde{\mu}}{\epsilon + \|\mathbf{x}'[n]\|^2}
e[n]\mathbf{x}'[n]
$$

The minus sign comes from the convention $e[n] = d[n] + y_s[n]$: the controller is adapting a signal that should destructively interfere with the disturbance. This notebook compares FxLMS and FxNLMS, then runs an HVAC-style case with fan harmonics and secondary-path mismatch.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(3)


## 1. FxLMS vs FxNLMS on the Same Physical ANC Model

Both algorithms use the same primary path $P(z)$ and secondary path $S(z)$. The difference is only the update rule: FxLMS uses a fixed step size, while FxNLMS normalizes by filtered-reference energy.


In [ ]:
N = 1000
n = np.arange(N)

level = 0.75 + 0.40 * (n / N)
x = level * (
    np.sin(2 * np.pi * 0.045 * n)
    + 0.55 * np.sin(2 * np.pi * 0.095 * n)
)

p = np.array([0.00, 0.00, 0.30, 0.70, 0.90, 0.60, 0.30, 0.10])
s_true = np.array([0.00, 0.00, 0.50, 0.80, 0.60, 0.30, 0.10])
s_hat = s_true.copy()

d = np.convolve(x, p, mode='full')[:N]
x_filt = np.convolve(x, s_hat, mode='full')[:N]

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=False)
fig.suptitle('Physical ANC Signal and Path Setup', fontsize=13)

axes[0].plot(n, x, label='Reference x[n]', color='tab:blue')
axes[0].set_ylabel('Amplitude')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(n, d, label='Primary disturbance d[n] = p * x[n]', color='tab:orange')
axes[1].set_ylabel('Amplitude')
axes[1].legend()
axes[1].grid(True)

axes[2].stem(np.arange(len(p)), p, basefmt=' ', label='Primary path p[n]')
axes[2].stem(np.arange(len(s_true)), s_true, linefmt='C2-', markerfmt='C2o', basefmt=' ', label='Secondary path s[n]')
axes[2].set_xlabel('Tap index')
axes[2].set_ylabel('Coefficient')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
M = 16
Ls = len(s_true)
pad = M + Ls + 5

x_buf = np.concatenate([np.zeros(pad), x])
d_buf = np.concatenate([np.zeros(pad), d])
xf_buf = np.concatenate([np.zeros(pad), x_filt])

# FxLMS implementation.
mu = 0.0003
w_fxlms = np.zeros(M)
y_buf_fxlms = np.zeros(Ls)
e_fxlms = np.zeros(N)
ys_fxlms = np.zeros(N)

for i in range(N):
    ii = i + pad
    x_vec = x_buf[ii : ii - M : -1]
    xf_vec = xf_buf[ii : ii - M : -1]

    y = np.dot(w_fxlms, x_vec)
    y_buf_fxlms[1:] = y_buf_fxlms[:-1]
    y_buf_fxlms[0] = y
    y_s = np.dot(s_true, y_buf_fxlms)

    e_fxlms[i] = d_buf[ii] + y_s
    ys_fxlms[i] = y_s
    w_fxlms = w_fxlms - mu * e_fxlms[i] * xf_vec

# FxNLMS implementation.
mu_tilde = 0.02
eps = 1e-6
w_fxnlms = np.zeros(M)
y_buf_fxnlms = np.zeros(Ls)
e_fxnlms = np.zeros(N)
ys_fxnlms = np.zeros(N)
mu_eff_history = np.zeros(N)

for i in range(N):
    ii = i + pad
    x_vec = x_buf[ii : ii - M : -1]
    xf_vec = xf_buf[ii : ii - M : -1]

    y = np.dot(w_fxnlms, x_vec)
    y_buf_fxnlms[1:] = y_buf_fxnlms[:-1]
    y_buf_fxnlms[0] = y
    y_s = np.dot(s_true, y_buf_fxnlms)

    e_fxnlms[i] = d_buf[ii] + y_s
    ys_fxnlms[i] = y_s
    mu_eff = mu_tilde / (np.dot(xf_vec, xf_vec) + eps)
    w_fxnlms = w_fxnlms - mu_eff * e_fxnlms[i] * xf_vec
    mu_eff_history[i] = mu_eff

steady = slice(N // 2, None)
reduction_fxlms = 10 * np.log10(np.var(d[steady]) / (np.var(e_fxlms[steady]) + 1e-12))
reduction_fxnlms = 10 * np.log10(np.var(d[steady]) / (np.var(e_fxnlms[steady]) + 1e-12))

print(f'FxLMS noise reduction:  {reduction_fxlms:.1f} dB')
print(f'FxNLMS noise reduction: {reduction_fxnlms:.1f} dB')


In [ ]:
mse_fxlms = np.convolve(e_fxlms**2, np.ones(35) / 35, mode='same')
mse_fxnlms = np.convolve(e_fxnlms**2, np.ones(35) / 35, mode='same')

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
fig.suptitle('Filtered-x ANC: FxLMS vs FxNLMS', fontsize=13)

axes[0].plot(n, d, label='Disturbance d[n]', color='tab:blue')
axes[0].plot(n, -ys_fxlms, label='FxLMS cancellation -y_s[n]', color='tab:orange', linestyle='--')
axes[0].plot(n, -ys_fxnlms, label='FxNLMS cancellation -y_s[n]', color='tab:green', linestyle=':')
axes[0].set_ylabel('Amplitude')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(10 * np.log10(mse_fxlms + 1e-12), label='FxLMS residual power', color='tab:orange')
axes[1].plot(10 * np.log10(mse_fxnlms + 1e-12), label='FxNLMS residual power', color='tab:green')
axes[1].set_ylabel('Residual power (dB)')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(mu_eff_history, label='FxNLMS effective step size', color='tab:purple')
axes[2].set_xlabel('Sample index n')
axes[2].set_ylabel('mu_eff[n]')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()


## 2. HVAC Duct Case Study

Fan noise is often harmonic and predictable, making ducts and exhaust paths natural ANC examples. This section uses a longer primary path, a longer secondary path, and a 10 percent random secondary-path modeling error.


In [ ]:
np.random.seed(7)

N = 1400
n = np.arange(N)
f0 = 0.055
level = 0.75 + 0.35 * (n / N)

x = level * (
    np.sin(2 * np.pi * f0 * n)
    + 0.6 * np.sin(2 * np.pi * 2 * f0 * n)
    + 0.3 * np.sin(2 * np.pi * 3 * f0 * n)
)

p = np.array([0.00, 0.00, 0.05, 0.18, 0.45, 0.85, 1.00, 0.75, 0.42, 0.20, 0.08])
s_true = np.array([0.00, 0.00, 0.12, 0.36, 0.72, 0.90, 0.58, 0.28, 0.10])

mismatch = 0.10 * np.random.randn(len(s_true))
s_hat = s_true * (1.0 + mismatch)

d = np.convolve(x, p, mode='full')[:N]
x_filt = np.convolve(x, s_hat, mode='full')[:N]

print('Secondary-path mismatch (% per tap):')
print(np.round(100 * mismatch, 1))


In [ ]:
M = 32
mu_tilde = 0.05
eps = 1e-6
Ls = len(s_true)
pad = M + Ls + 5

x_buf = np.concatenate([np.zeros(pad), x])
d_buf = np.concatenate([np.zeros(pad), d])
xf_buf = np.concatenate([np.zeros(pad), x_filt])

w = np.zeros(M)
y_buf = np.zeros(Ls)
e = np.zeros(N)
y_secondary = np.zeros(N)

for i in range(N):
    ii = i + pad
    x_vec = x_buf[ii : ii - M : -1]
    xf_vec = xf_buf[ii : ii - M : -1]

    y = np.dot(w, x_vec)
    y_buf[1:] = y_buf[:-1]
    y_buf[0] = y
    y_s = np.dot(s_true, y_buf)

    e[i] = d_buf[ii] + y_s
    y_secondary[i] = y_s

    mu_eff = mu_tilde / (np.dot(xf_vec, xf_vec) + eps)
    w = w - mu_eff * e[i] * xf_vec

steady = slice(N // 2, None)
reduction_db = 10 * np.log10(np.var(d[steady]) / (np.var(e[steady]) + 1e-12))
print(f'HVAC FxNLMS noise reduction with path mismatch: {reduction_db:.1f} dB')


In [ ]:
mse = np.convolve(e**2, np.ones(55) / 55, mode='same')

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
fig.suptitle('HVAC Duct ANC - FxNLMS with Secondary-Path Mismatch', fontsize=13)

axes[0].plot(n, x, label='Fan harmonic reference x[n]', color='tab:blue')
axes[0].set_ylabel('Amplitude')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(n, d, label='Duct disturbance d[n]', color='tab:orange')
axes[1].plot(n, -y_secondary, label='Cancellation reaching error mic -y_s[n]', color='tab:green', linestyle='--')
axes[1].plot(n, e, label='Residual e[n]', color='tab:red', alpha=0.7)
axes[1].set_ylabel('Amplitude')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(10 * np.log10(mse + 1e-12), color='tab:purple')
axes[2].set_xlabel('Sample index n')
axes[2].set_ylabel('Residual power (dB)')
axes[2].set_title('Learning curve')
axes[2].grid(True)

plt.tight_layout()
plt.show()
